In [1]:

import sqlite3
import random
import time
import requests
import pandas as pd

random.seed(42)


In [2]:

conn = sqlite3.connect("school.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    major TEXT,
    gpa REAL
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course TEXT,
    grade TEXT,
    FOREIGN KEY (student_id) REFERENCES students(student_id)
)
""")

conn.commit()


In [3]:

students_data = [
    (1, "Alice", "CS", 3.8),
    (2, "Bob", "CS", 3.6),
    (3, "Charlie", "IT", 3.2),
    (4, "David", "CS", 3.9),
    (5, "Emma", "IT", 3.7),
    (6, "Frank", "CS", 3.4),
    (7, "Grace", "AI", 3.9),
    (8, "Henry", "AI", 3.5)
]

cur.executemany("""
INSERT OR IGNORE INTO students
(student_id, name, major, gpa)
VALUES (?, ?, ?, ?)
""", students_data)

conn.commit()

print("Student data inserted successfully!")


Student data inserted successfully!


In [4]:

enrollments_data = [
    (1, 1, "Python", "A"),
    (2, 1, "Database", "A"),
    (3, 2, "Python", "B"),
    (4, 3, "Networking", "B"),
    (5, 4, "Database", "A"),
    (6, 5, "Python", "B"),
    (7, 6, "Networking", "C"),
    (8, 7, "AI", "A")
]

cur.executemany("""
INSERT OR IGNORE INTO enrollments
(enrollment_id, student_id, course, grade)
VALUES (?, ?, ?, ?)
""", enrollments_data)

conn.commit()

print("Enrollment data inserted successfully!")


Enrollment data inserted successfully!


In [5]:

query = """
SELECT name, gpa
FROM students
WHERE major = 'CS' AND gpa >= 3.5
ORDER BY gpa DESC
"""

print(pd.read_sql(query, conn))


    name  gpa
0  David  3.9
1  Alice  3.8
2    Bob  3.6


In [6]:

query = """
SELECT students.name, enrollments.course, enrollments.grade
FROM students
INNER JOIN enrollments
ON students.student_id = enrollments.student_id
"""

print(pd.read_sql(query, conn))


      name      course grade
0    Alice      Python     A
1    Alice    Database     A
2      Bob      Python     B
3  Charlie  Networking     B
4    David    Database     A
5     Emma      Python     B
6    Frank  Networking     C
7    Grace          AI     A


In [7]:

query = """
SELECT major,
       COUNT(*) AS n_students,
       AVG(gpa) AS avg_gpa
FROM students
GROUP BY major
HAVING COUNT(*) > 2
ORDER BY avg_gpa DESC
"""

print(pd.read_sql(query, conn))


  major  n_students  avg_gpa
0    CS           4    3.675


In [8]:

# UNSAFE: Do not build SQL queries using user input and f-strings
unsafe_name = "CS"

# SAFE: Use ? placeholders
cur.execute(
    "SELECT name, gpa FROM students WHERE major = ?",
    ("CS",)
)

rows = cur.fetchall()
print(rows)


[('Alice', 3.8), ('Bob', 3.6), ('David', 3.9), ('Frank', 3.4)]


In [9]:

query = """
SELECT major, AVG(gpa) AS avg_gpa
FROM students
GROUP BY major
"""

df = pd.read_sql(query, conn)
print(df)


  major  avg_gpa
0    AI    3.700
1    CS    3.675
2    IT    3.450


In [10]:

conn.close()

print("Database connection closed.")


Database connection closed.


In [11]:

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 37.5665,
    "longitude": 126.9780,
    "current_weather": True
}

try:
    response = requests.get(
        url,
        params=params,
        timeout=10
    )

    print("Status code:", response.status_code)
    response.raise_for_status()

    data = response.json()
    print(data["current_weather"])

except requests.exceptions.RequestException as e:
    print("Could not fetch live weather:", e)

    data = {
        "current_weather": {
            "temperature": 18.4,
            "windspeed": 9.1
        }
    }


Status code: 200
{'time': '2026-10-13T12:45', 'interval': 900, 'temperature': 15.5, 'windspeed': 9.4, 'winddirection': 302, 'is_day': 0, 'weathercode': 0}


In [12]:

weather_df = pd.DataFrame([data["current_weather"]])

print(weather_df)


               time  interval  temperature  windspeed  winddirection  is_day  \
0  2026-10-13T12:45       900         15.5        9.4            302       0   

   weathercode  
0            0  


In [13]:

from bs4 import BeautifulSoup

url = "https://books.toscrape.com"

headers = {
    "User-Agent": "KDU-DataScience-Student-Demo"
}

response = requests.get(
    url,
    headers=headers,
    timeout=10
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

titles = [
    h.get_text(strip=True)
    for h in soup.select("h3 a")
]

print(titles[:5])

time.sleep(1)  # Wait politely before another request


['A Light in the ...', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History ...']


In [15]:

books_df = pd.DataFrame(titles, columns=["Book Title"])

print(books_df.head())


                     Book Title
0            A Light in the ...
1            Tipping the Velvet
2                    Soumission
3                 Sharp Objects
4  Sapiens: A Brief History ...
